In [ ]:
# importing necessary libraries
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Lists

In [ ]:
# Creating lists
list_a = [1, 2, 3, 4, 5]
list_b = [6, 7, 8, 9, 10]

In [ ]:
# Operations on lists
# Adding lists
list_sum = [a + b for a, b in zip(list_a, list_b)]
print("List Sum:", list_sum)

# Vector product using lists    
vector_product = [a * b for a, b in zip(list_a, list_b)]
print("Vector Product:", vector_product)

# Numpy Array

In [ ]:
# Creating numpy arrays
numpy_array_a = np.array(list_a)
numpy_array_b = np.array(list_b)

In [ ]:
# Operations on numpy arrays
# Adding numpy arrays
numpy_sum = numpy_array_a + numpy_array_b
print("Numpy Sum:", numpy_sum)

# Vector product using numpy arrays
numpy_vector_product = np.multiply(numpy_array_a, numpy_array_b)
print("Numpy Vector Product:", numpy_vector_product)

In [ ]:
np.allclose(list_sum, numpy_sum), np.allclose(vector_product, numpy_vector_product)

# Time comparison between list and numpy array

In [ ]:
# Creating large arrays and lists for time comparison
numpy_array_a = np.random.randint(0, 100, size=10000)
numpy_array_b = np.random.randint(0, 100, size=10000)

list_a = list(numpy_array_a)
list_b = list(numpy_array_b)

In [ ]:
# Time for list addition
start_time = time.time()
for _ in range(1000):
    list_sum = [a + b for a, b in zip(list_a, list_b)]
end_time = time.time()
print("Time taken for lists addition:", end_time - start_time)

# Time for numpy addition
start_time = time.time()
for _ in range(1000):
    numpy_sum = numpy_array_a + numpy_array_b
end_time = time.time()
print("Time taken for numpy addition:", end_time - start_time)

In [ ]:
# Time for list vector product
start_time = time.time()
for _ in range(10000):
    list_product = [a * b for a, b in zip(list_a, list_b)]

end_time = time.time()
print("Time taken for list vector product:", end_time - start_time)

# Time for numpy vector product 
start_time = time.time()
for _ in range(10000):
    numpy_product = np.multiply(numpy_array_a, numpy_array_b)

end_time = time.time()
print("Time taken for numpy vector product:", end_time - start_time)

In [ ]:
np.allclose(list_sum, numpy_sum), np.allclose(vector_product, numpy_vector_product)

In [ ]:
timeit_add_list = %timeit -o [a + b for a, b in zip(list_a, list_b)]

In [ ]:
timeit_add_numpy = %timeit -o numpy_array_a + numpy_array_b

# Code clarity

In [ ]:
# Numpy code is often more concise and readable than list comprehensions
# Example: Calculate the element-wise product of two lists
list_product = [a * b for a, b in zip(list_a, list_b)]
numpy_product = np.multiply(numpy_array_a, numpy_array_b)

In [ ]:
numpy_product

In [ ]:
numpy_array_a@numpy_array_b

# Reading CSV file using Numpy


In [ ]:
!head ../datasets/tennis-discrete-output.csv

In [ ]:
np.genfromtxt?

In [ ]:
data = np.genfromtxt('../datasets/tennis-discrete-output.csv', delimiter=',')
data

Wait! What happened?

In [ ]:
data = np.genfromtxt('../datasets/tennis-discrete-output.csv', delimiter=',', dtype=str)
data

In [ ]:
data.shape

Question: Find the outlook on D11


In [ ]:
idx = np.argwhere(data[:, 0] == 'D11')[0, 0]
idx

In [ ]:
data[idx]

In [ ]:
data[idx][1]

# Reading CSV file using Pandas

In [ ]:
df = pd.read_csv('../datasets/tennis-discrete-output.csv')

In [ ]:
df

In [ ]:
df['Day'] == 'D11'

In [ ]:
df[df['Day'] == 'D11']

In [ ]:
df[df['Day'] == 'D11']['Outlook']

In [ ]:
df.query('Day == "D11"')['Outlook']

In [ ]:
df.shape

Question. How many times do we play v/s not play tennis

In [ ]:
ser = df['Play']
ser

In [ ]:
unique_play_options = df['Play'].unique()
unique_play_options

In [ ]:
for option in unique_play_options:
    print(option, (df['Play'] == option).sum())


In [ ]:
df['Play'].value_counts()

In [ ]:
df.groupby('Play').size()

In [ ]:
gby = df.groupby('Play')

In [ ]:
{k: len(v) for k, v in gby.groups.items()}

In [ ]:
pd.crosstab(index=df['Play'], columns='count')

What is the distribution of any given attribute?

In [ ]:
def distribution(df, attribute):
    return df[attribute].value_counts()

In [ ]:
ser = distribution(df, 'Outlook')

In [ ]:
ser

In [ ]:
type(ser)

In [ ]:
ser.values

In [ ]:
ser.index

In [ ]:
distribution(df, 'Temp')

Finding entropy for target variable

In [ ]:
target_attribute = 'Play'
dist_target = distribution(df, target_attribute)

In [ ]:
dist_target

Normalize distribution

In [ ]:
dist_target/dist_target.sum()

In [ ]:
df['Play'].value_counts(normalize=True)

In [ ]:
normalized_dist_target = dist_target/dist_target.sum()

For loop way of calculating entropy

In [ ]:
e = 0.0
for value, p in normalized_dist_target.items():
    e = e - p * np.log2(p + 1e-6) # 1e-6 is added to avoid log(0)
print(e)

In [ ]:
normalized_dist_target.apply(lambda x: -x * np.log2(x + 1e-6))

In [ ]:
normalized_dist_target.apply(lambda x: -x * np.log2(x + 1e-6)).sum()

More on crosstab

In [ ]:
pd.crosstab(index=df['Outlook'], columns=df['Play'])

In [ ]:
pd.crosstab(index=df['Outlook'], columns=df['Play']).T

In [ ]:
df_attr = pd.crosstab(index=df['Play'], columns=df['Outlook'], normalize='columns')
df_attr

Using groupby

In [ ]:
df.groupby(['Play', 'Outlook']).size()

In [ ]:
df.groupby(['Play', 'Outlook']).size().index

In [ ]:
df.groupby(['Play', 'Outlook']).size().unstack('Outlook')

In [ ]:
df_attr_groupby = df.groupby(['Play', 'Outlook']).size().unstack('Outlook').fillna(0)
df_attr_groupby

Apply

In [ ]:
neg_plogp = df_attr.apply(lambda x: -x * np.log2(x + 1e-6), axis=0)
neg_plogp

In [ ]:
neg_plogp.sum(axis=0).sort_index()

In [ ]:
df_attr_dist = distribution(df, 'Outlook')
norm_attr_dist = df_attr_dist/df_attr_dist.sum()
norm_attr_dist

In [ ]:
(norm_attr_dist*neg_plogp.sum(axis=0).sort_index()).sum()